In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt

# =========================
# PATHS
# =========================
BASE_DIR = Path(r"C:\Plegma_Programming")
DATA_FILE = BASE_DIR / "PLEGMA_final_hourly_dataset.csv"

OUTPUT_DIR = BASE_DIR / "Results" / "Baselines"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CSV_OUT = OUTPUT_DIR / "baseline_results.csv"
PNG_MAE = OUTPUT_DIR / "baseline_mae.png"
PNG_RMSE = OUTPUT_DIR / "baseline_rmse.png"
PNG_MAPE = OUTPUT_DIR / "baseline_mape.png"

# =========================
# LOAD
# =========================
df = pd.read_csv(DATA_FILE)
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values(["home_id", "timestamp"]).reset_index(drop=True)

# =========================
# CREATE LAGS
# =========================
df["lag_1"] = df.groupby("home_id")["consumption_Wh"].shift(1)
df["lag_24"] = df.groupby("home_id")["consumption_Wh"].shift(24)

# =========================
# TIME SPLIT
# =========================
split_index = int(len(df) * 0.8)

train = df.iloc[:split_index].copy()
test = df.iloc[split_index:].copy()

print("Train shape:", train.shape)
print("Test shape :", test.shape)

# =========================
# HELPERS
# =========================
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def mape(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    return np.mean(np.abs((y_true - y_pred) / (y_true + 1e-5))) * 100

def evaluate_baseline(name, y_true, y_pred):
    mask = (~pd.isna(y_true)) & (~pd.isna(y_pred))
    y_true_valid = y_true[mask]
    y_pred_valid = y_pred[mask]

    return {
        "baseline": name,
        "n_samples": len(y_true_valid),
        "MAE": mean_absolute_error(y_true_valid, y_pred_valid),
        "RMSE": rmse(y_true_valid, y_pred_valid),
        "MAPE": mape(y_true_valid, y_pred_valid),
    }

# =========================
# BASELINE 01
# Previous day, same hour (lag_24)
# =========================
y_true = test["consumption_Wh"]
y_pred_b1 = test["lag_24"]

# =========================
# BASELINE 02
# Global mean by hour
# =========================
global_hour_mean = train.groupby("hour")["consumption_Wh"].mean().to_dict()
y_pred_b2 = test["hour"].map(global_hour_mean)

# =========================
# BASELINE 03
# Mean by hour + weekday/weekend
# =========================
global_hour_weekend_mean = (
    train.groupby(["hour", "is_weekend"])["consumption_Wh"]
    .mean()
    .to_dict()
)

y_pred_b3 = test.apply(
    lambda row: global_hour_weekend_mean.get((row["hour"], row["is_weekend"]), np.nan),
    axis=1
)

# =========================
# BASELINE 04
# Per-home mean by hour
# =========================
per_home_hour_mean = (
    train.groupby(["home_id", "hour"])["consumption_Wh"]
    .mean()
    .to_dict()
)

y_pred_b4 = test.apply(
    lambda row: per_home_hour_mean.get((row["home_id"], row["hour"]), np.nan),
    axis=1
)

# =========================
# BASELINE 05
# Persistence (lag_1)
# =========================
y_pred_b5 = test["lag_1"]

# =========================
# EVALUATE
# =========================
results = []
results.append(evaluate_baseline("Baseline 01 - Previous day same hour (lag_24)", y_true, y_pred_b1))
results.append(evaluate_baseline("Baseline 02 - Global mean by hour", y_true, y_pred_b2))
results.append(evaluate_baseline("Baseline 03 - Mean by hour + weekend", y_true, y_pred_b3))
results.append(evaluate_baseline("Baseline 04 - Per-home mean by hour", y_true, y_pred_b4))
results.append(evaluate_baseline("Baseline 05 - Persistence (lag_1)", y_true, y_pred_b5))

results_df = pd.DataFrame(results).sort_values("MAE").reset_index(drop=True)

print("\nBaseline Results:")
print(results_df)

# =========================
# SAVE CSV
# =========================
results_df.to_csv(CSV_OUT, index=False)
print("\nSaved CSV:", CSV_OUT)

# =========================
# PLOTS
# =========================
def save_metric_plot(df_results, metric, out_path):
    plot_df = df_results.sort_values(metric, ascending=True)

    plt.figure(figsize=(11, 6))
    plt.barh(plot_df["baseline"], plot_df[metric])
    plt.xlabel(metric)
    plt.title(f"PLEGMA Baseline Comparison - {metric}")
    plt.tight_layout()
    plt.savefig(out_path, dpi=300)
    plt.close()

save_metric_plot(results_df, "MAE", PNG_MAE)
save_metric_plot(results_df, "RMSE", PNG_RMSE)
save_metric_plot(results_df, "MAPE", PNG_MAPE)

print("Saved plot:", PNG_MAE)
print("Saved plot:", PNG_RMSE)
print("Saved plot:", PNG_MAPE)

Train shape: (56888, 37)
Test shape : (14223, 37)

Baseline Results:
                                        baseline  n_samples         MAE  \
0              Baseline 05 - Persistence (lag_1)      14221  249.988413   
1            Baseline 04 - Per-home mean by hour       1852  327.682414   
2              Baseline 02 - Global mean by hour      14223  343.228289   
3           Baseline 03 - Mean by hour + weekend      14223  343.614563   
4  Baseline 01 - Previous day same hour (lag_24)      14175  364.819648   

         RMSE        MAPE  
0  528.533705   62.736241  
1  540.967934  117.561446  
2  611.726376   98.117678  
3  612.211395   98.406239  
4  693.200183  115.401439  

Saved CSV: C:\Plegma_Programming\Results\Baselines\baseline_results.csv
Saved plot: C:\Plegma_Programming\Results\Baselines\baseline_mae.png
Saved plot: C:\Plegma_Programming\Results\Baselines\baseline_rmse.png
Saved plot: C:\Plegma_Programming\Results\Baselines\baseline_mape.png
